In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    upper,
    lower,
    to_date,
    to_timestamp,
    when,
    current_timestamp,
    from_json
)

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DecimalType,
    IntegerType
)


# ============================================
# STORAGE PATHS
# ============================================

bronze_path = (
    "abfss://bronze@bankingdelakevishal.dfs.core.windows.net/transaction/"
)

silver_path = (
    "abfss://silver@bankingdelakevishal.dfs.core.windows.net/transaction/"
)

In [0]:
# ============================================
# TRANSACTION: BRONZE → SILVER
# CELL 2 - READ, PARSE AND CLEAN DATA
# ============================================

# --------------------------------------------
# STORAGE & CATALOG CONFIGURATION
# --------------------------------------------
CATALOG_NAME = "banking_lakehouse_db2"
SCHEMA_NAME = "silver"
TABLE_NAME = "transaction"
FULL_TABLE_NAME = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.{TABLE_NAME}"

bronze_path = "abfss://bronze@bankingdelakevishal.dfs.core.windows.net/transaction/"
silver_path = "abfss://silver@bankingdelakevishal.dfs.core.windows.net/transaction/"

# Ensure Silver schema exists inside Unity Catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG_NAME}`.{SCHEMA_NAME}")

# --------------------------------------------
# TRANSACTION JSON SCHEMA
# --------------------------------------------
transaction_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("account_id", StringType(), True),
    StructField("transaction_timestamp", StringType(), True),
    StructField("transaction_type", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("merchant", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("status", StringType(), True),
    StructField("fraud_flag", StringType(), True)
])

# --------------------------------------------
# READ BRONZE TRANSACTION DATA
# --------------------------------------------
transaction_bronze = spark.read.parquet(bronze_path)
bronze_count = transaction_bronze.count()
print(f"Bronze Record Count: {bronze_count}")

# --------------------------------------------
# PARSE JSON
# --------------------------------------------
transaction_parsed = (
    transaction_bronze
    .withColumn("parsed_data", from_json(col("transaction_data"), transaction_schema))
    .select(
        trim(col("parsed_data.transaction_id")).alias("transaction_id"),
        trim(col("parsed_data.account_id")).alias("account_id"),
        trim(col("parsed_data.transaction_timestamp")).alias("transaction_timestamp_raw"),
        upper(trim(col("parsed_data.transaction_type"))).alias("transaction_type"),
        upper(trim(col("parsed_data.channel"))).alias("channel"),
        trim(col("parsed_data.merchant")).alias("merchant"),
        trim(col("parsed_data.amount")).cast(DecimalType(18, 2)).alias("amount"),
        upper(trim(col("parsed_data.currency"))).alias("currency"),
        upper(trim(col("parsed_data.status"))).alias("transaction_status"),
        upper(trim(col("parsed_data.fraud_flag"))).alias("fraud_flag_raw"),
        # Event Hub metadata
        col("topic"),
        col("partition"),
        col("offset"),
        col("timestamp").alias("event_hub_timestamp")
    )
)

# --------------------------------------------
# CONVERT TIMESTAMP + FRAUD FLAG
# --------------------------------------------
transaction_cleaned = (
    transaction_parsed
    .withColumn("transaction_timestamp", to_timestamp(col("transaction_timestamp_raw")))
    .withColumn(
        "fraud_flag",
        when(col("fraud_flag_raw") == "Y", True)
        .when(col("fraud_flag_raw") == "N", False)
        .otherwise(None)
    )
    .drop("transaction_timestamp_raw", "fraud_flag_raw")
)

# --------------------------------------------
# DATA QUALITY FILTERING
# --------------------------------------------
transaction_valid = (
    transaction_cleaned
    # Required transaction ID & Account ID
    .filter(col("transaction_id").isNotNull() & (trim(col("transaction_id")) != ""))
    .filter(col("account_id").isNotNull() & (trim(col("account_id")) != ""))
    
    # Valid timestamp & positive amount
    .filter(col("transaction_timestamp").isNotNull())
    .filter(col("amount").isNotNull() & (col("amount") > 0))
    
    # Required fields & fraud flag check
    .filter(col("transaction_type").isNotNull())
    .filter(col("transaction_status").isNotNull())
    .filter(col("fraud_flag").isNotNull())
    
    # Deduplication & metadata
    .dropDuplicates(["transaction_id"])
    .withColumn("silver_ingestion_timestamp", current_timestamp())
)

# --------------------------------------------
# METRICS SUMMARY
# --------------------------------------------
valid_count = transaction_valid.count()
print(f"Valid Silver Record Count: {valid_count}")
print(f"Dropped / Duplicate Records: {bronze_count - valid_count}")

In [0]:
# ============================================
# TRANSACTION: BRONZE → SILVER
# CELL 3 - WRITE AND VALIDATE
# ============================================

# --------------------------------------------
# WRITE TO SILVER (ADLS PATH + CATALOG TABLE)
# --------------------------------------------
# 1. Overwrite raw Delta files in ADLS
(
    transaction_valid
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_path)
)

# 2. Overwrite / Register managed Unity Catalog table
(
    transaction_valid
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FULL_TABLE_NAME)
)

print(f"Transaction data successfully loaded to Silver path and catalog table '{FULL_TABLE_NAME}'.")


# --------------------------------------------
# READ SILVER FOR VALIDATION FROM CATALOG
# --------------------------------------------
transaction_silver = spark.table(FULL_TABLE_NAME)

print(
    "Final Silver Record Count:",
    transaction_silver.count()
)


# --------------------------------------------
# CHECK SCHEMA
# --------------------------------------------
transaction_silver.printSchema()


# --------------------------------------------
# VIEW DATA
# --------------------------------------------
display(
    transaction_silver.limit(10)
)